In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

order_items_features = pd.read_csv(PROCESSED_DIR / "order_items_features.csv")

print(order_items.shape)
print(order_items.columns.tolist())

In [ ]:
print(order_items[['order_id', 'product_id']].head())

In [ ]:
from pathlib import Path
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

orders_features = pd.read_csv(PROCESSED_DIR / "orders_features.csv")

print(orders[['order_id', 'customer_id']].head())

In [ ]:
purchase_data = orders[['order_id', 'customer_id']].merge(
    order_items[['order_id', 'product_id']],
    on='order_id',
    how='inner'
)

print(purchase_data.shape)
print(purchase_data.head())

In [ ]:
purchase_data = purchase_data.drop_duplicates(
    subset=['customer_id', 'product_id']
)

print(purchase_data.shape)

In [ ]:
print(purchase_data['customer_id'].nunique())
print(purchase_data['product_id'].nunique())

In [ ]:
from scipy.sparse import csr_matrix

In [ ]:
# Remove duplicates
purchase_data = purchase_data.drop_duplicates(
    subset=['customer_id', 'product_id']
).copy()

# Convert customer IDs and product IDs into integer indices
customer_codes, customer_uniques = pd.factorize(
    purchase_data['customer_id']
)

product_codes, product_uniques = pd.factorize(
    purchase_data['product_id']
)

# Create sparse Customer × Product matrix
customer_product_sparse = csr_matrix(
    (
        np.ones(len(purchase_data)),
        (customer_codes, product_codes)
    ),
    shape=(
        len(customer_uniques),
        len(product_uniques)
    )
)

print("Matrix shape:", customer_product_sparse.shape)
print("Non-zero purchases:", customer_product_sparse.nnz)

In [ ]:
product_customer_sparse = customer_product_sparse.T.tocsr()

print("Product × Customer shape:")
print(product_customer_sparse.shape)

In [ ]:
product_ids = list(product_uniques)

product_index = {
    product_id: index
    for index, product_id in enumerate(product_ids)
}

print("Number of products:", len(product_ids))

In [ ]:
from sklearn.neighbors import NearestNeighbors

knn_model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=6,
    n_jobs=-1
)

knn_model.fit(product_customer_sparse)

In [ ]:
def recommend_products(product_id, n=5):

    if product_id not in product_index:
        return f"Product ID '{product_id}' not found."

    idx = product_index[product_id]

    distances, indices = knn_model.kneighbors(
        product_customer_sparse[idx],
        n_neighbors=n + 1
    )

    recommendations = []

    for distance, index in zip(
        distances[0],
        indices[0]
    ):

        # Skip the original product
        if index == idx:
            continue

        similarity = 1 - distance

        recommendations.append({
            'product_id': product_ids[index],
            'similarity_score': similarity
        })

    return pd.DataFrame(recommendations)

In [ ]:
sample_product = product_ids[0]

print("Input Product:")
print(sample_product)

In [ ]:
recommendations = recommend_products(
    sample_product,
    n=5
)

recommendations

In [ ]:
recommendations

In [ ]:
popular_products = (
    purchase_data['product_id']
    .value_counts()
    .head(20)
)

popular_products

In [ ]:
sample_product = popular_products.index[0]

print("Popular product:")
print(sample_product)

In [ ]:
recommendations = recommend_products(
    sample_product,
    n=5
)

recommendations

In [ ]:
products = pd.read_csv(
    "../data/processed/products_features.csv"
)

print(products.columns.tolist())

In [ ]:
recommendations = recommendations.merge(
    products[
        ['product_id', 'product_category_name']
    ],
    on='product_id',
    how='left'
)

recommendations

In [ ]:
recommendations.insert(
    0,
    'source_product_id',
    sample_product
)

recommendations

In [ ]:
recommendations.to_csv(
    "../outputs/recommendations.csv",
    index=False
)

print("Recommendation file saved successfully.")

In [ ]:
all_recommendations = []

top_products = (
    purchase_data['product_id']
    .value_counts()
    .head(100)
    .index
)

for product_id in top_products:

    recs = recommend_products(
        product_id,
        n=5
    )

    if isinstance(recs, pd.DataFrame):

        recs['source_product_id'] = product_id

        all_recommendations.append(recs)

In [ ]:
recommendation_df = pd.concat(
    all_recommendations,
    ignore_index=True
)

print(recommendation_df.shape)
print(recommendation_df.head())

In [ ]:
recommendation_df = recommendation_df.merge(
    products[
        ['product_id', 'product_category_name']
    ],
    on='product_id',
    how='left'
)

recommendation_df.head()

In [ ]:
recommendation_df.to_csv(
    "../outputs/product_recommendations.csv",
    index=False
)

print("Final recommendation dataset saved.")

In [ ]:
print("Matrix shape:", customer_product_sparse.shape)
print("Non-zero purchases:", customer_product_sparse.nnz)
print("Product × Customer shape:", product_customer_sparse.shape)

In [ ]:
recommendation_df = pd.read_csv(
    "../outputs/product_recommendations.csv"
)

print("Shape:", recommendation_df.shape)
recommendation_df.head(10)

In [ ]:
print(recommendation_df.columns.tolist())

In [ ]:
recommendation_df["similarity_score"].describe()

In [ ]:
recommendation_df.sort_values(
    "similarity_score",
    ascending=False
).head(10)

In [ ]:
# Remove self-recommendations
recommendation_df = recommendation_df[
    recommendation_df["product_id"] != recommendation_df["source_product_id"]
].copy()

# Remove zero-similarity recommendations
recommendation_df = recommendation_df[
    recommendation_df["similarity_score"] > 0
].copy()

print("Recommendations after filtering:", recommendation_df.shape)

In [ ]:
recommendation_df["similarity_score"].describe()

In [ ]:
recommendation_df = (
    recommendation_df
    .sort_values(
        ["source_product_id", "similarity_score"],
        ascending=[True, False]
    )
    .groupby("source_product_id")
    .head(5)
    .reset_index(drop=True)
)

print("Final recommendation shape:", recommendation_df.shape)

In [ ]:
recommendation_df.head(20)

In [ ]:
products_with_recommendations = (
    recommendation_df["source_product_id"]
    .nunique()
)

total_products = purchase_data["product_id"].nunique()

print("Products with recommendations:", products_with_recommendations)
print("Total products:", total_products)

print(
    "Coverage:",
    round(products_with_recommendations / total_products * 100, 2),
    "%"
)

In [ ]:
final_recommendations = recommendation_df[
    [
        "source_product_id",
        "product_id",
        "similarity_score",
        "product_category_name"
    ]
].copy()

final_recommendations = final_recommendations.rename(
    columns={
        "source_product_id": "source_product_id",
        "product_id": "recommended_product_id",
        "similarity_score": "similarity_score",
        "product_category_name": "recommended_category"
    }
)

final_recommendations.head(10)

In [ ]:
final_recommendations.to_csv(
    "../outputs/final_product_recommendations.csv",
    index=False
)

print("Final recommendation dataset saved successfully.")

In [ ]:
print("Final recommendation shape:", final_recommendations.shape)
print("Products with recommendations:", products_with_recommendations)
print("Total products:", total_products)
print("Coverage:", round(products_with_recommendations / total_products * 100, 2), "%")